# BDC 2026 - Validasi Robustness Kalibrasi Champion (Multi-Seed)

Cek apakah kenaikan F1 0,9799 -> 0,9859 dari kalibrasi champion (2-model) itu **konsisten**, atau
cuma kebetulan beruntung di satu split tuning/holdout (seed=42). Proses split + grid search +
evaluasi holdout diulang puluhan kali dengan seed berbeda-beda.

**Cara baca hasilnya:**
- Kalau kalibrasi menang di HAMPIR SEMUA seed (bukan cuma sebagian) -> kenaikannya genuine, aman dipakai.
- Kalau menang-kalahnya cuma 50-50 atau lebih sering kalah -> 0,9859 kemarin itu kebetulan, JANGAN
  jadikan submission final berdasarkan itu.

In [18]:
import os

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split

In [19]:
CONFIG = {
    "root": "BDC 2026",
    "solution_csv": "solution.csv",
    "submission_template": "BDC 2026/submission.csv",
    "meta_model_path": "meta_model.pkl",           # <- pastikan ini meta-learner juara (v2)
    "test_probs_partial_path": "test_probs_partial.npy",  # cache test probs [convnextv2_tiny, siglip2]
    "submission_out": "submission_champion_calibrated.csv",
    "holdout_frac": 0.5,   # separuh solution.csv buat cari bobot, separuh buat validasi
    "seed": 42,
}

## Deteksi Kelas & Test DataFrame

In [20]:
def get_class_order(config):
    train_dir = os.path.join(config["root"], "train")
    return sorted(os.listdir(train_dir))


def build_test_dataframe(config):
    test_dir = os.path.join(config["root"], "test")
    test_images = sorted(
        os.listdir(test_dir),
        key=lambda x: int("".join(filter(str.isdigit, x)))
    )
    test_df = pd.DataFrame({"image": test_images})
    test_df["id"] = test_df["image"].apply(lambda x: int("".join(filter(str.isdigit, x))))
    return test_df


class_order = get_class_order(CONFIG)
print("Label mapping:", {name: i for i, name in enumerate(class_order)})

test_df = build_test_dataframe(CONFIG)

Label mapping: {'0_Recyclable': 0, '1_Electronic': 1, '2_Organic': 2}


## Hitung Probabilitas Test dari Meta-Learner Juara

Bukan cuma prediksi akhirnya (`predict`), tapi probabilitasnya (`predict_proba`) -- ini yang
dibutuhkan untuk kalibrasi.

In [21]:
meta_model = joblib.load(CONFIG["meta_model_path"])

test_probs_partial = np.load(CONFIG["test_probs_partial_path"], allow_pickle=True)
test_probs_list = [np.asarray(p, dtype=np.float64) for p in test_probs_partial]

X_meta_test = np.concatenate(test_probs_list, axis=1)
probs_test = meta_model.predict_proba(X_meta_test)  # shape (n_test, 3)

print(f"Shape probabilitas test: {probs_test.shape}")
print(f"Model classes_: {meta_model.classes_}")

Shape probabilitas test: (1458, 3)
Model classes_: [0 1 2]


## Gabungkan dengan solution.csv

In [22]:
assert os.path.exists(CONFIG["solution_csv"]), (
    f'Tidak ketemu \'{CONFIG["solution_csv"]}\' di direktori kerja saat ini '
    f'({os.getcwd()}). Cek lagi apakah file ini ada persis di folder itu.'
)
gt = pd.read_csv(CONFIG["solution_csv"])
assert "predicted" in gt.columns and "id" in gt.columns, (
    f"solution.csv kebaca, tapi kolomnya tidak sesuai ekspektasi (id, predicted). "
    f"Kolom yang ada sekarang: {gt.columns.tolist()}"
)
gt["predicted"] = gt["predicted"].fillna(0).astype(int)
gt = gt.rename(columns={"predicted": "true_label"})

eval_df = test_df[["id"]].copy()
eval_df["row_idx"] = np.arange(len(test_df))
eval_df = eval_df.merge(gt, on="id", how="inner")

y_true_all = eval_df["true_label"].values
probs_all = probs_test[eval_df["row_idx"].values]

print(f"Total baris dengan label manual: {len(eval_df)}")

Total baris dengan label manual: 1458


## Fungsi Evaluasi & Grid Search (sama seperti versi champion)

In [23]:
def macro_f1_with_weights(w, probs, y_true):
    adjusted = probs * w
    preds = adjusted.argmax(axis=1)
    return f1_score(y_true, preds, average="macro")


def grid_search_best_weights(probs_tuning, y_tuning, w_range):
    best_f1 = -1
    best_weights = np.array([1.0, 1.0, 1.0])
    for w0 in w_range:
        for w2 in w_range:
            w = np.array([w0, 1.0, w2])
            f1 = macro_f1_with_weights(w, probs_tuning, y_tuning)
            if f1 > best_f1:
                best_f1 = f1
                best_weights = w
    return best_weights, best_f1

## Loop Multi-Seed

Ulangi split tuning/holdout 50x dengan seed berbeda-beda. Tiap iterasi: cari bobot terbaik di
tuning, lalu evaluasi bobot itu (dan baseline tanpa kalibrasi) di holdout -- supaya adil, holdout
TIDAK PERNAH ikut proses pencarian bobot.

In [ ]:
w_range = np.linspace(0.05, 5.0, 120)  # dilebarkan dari 0.3-3.0 -- w_organic kemarin mentok di batas bawah
n_repeats = 50

records = []
for seed in range(n_repeats):
    tuning_idx, holdout_idx = train_test_split(
        np.arange(len(eval_df)),
        test_size=CONFIG["holdout_frac"],
        stratify=y_true_all,
        random_state=seed,
    )
    y_tuning, probs_tuning = y_true_all[tuning_idx], probs_all[tuning_idx]
    y_holdout, probs_holdout = y_true_all[holdout_idx], probs_all[holdout_idx]

    best_weights, _ = grid_search_best_weights(probs_tuning, y_tuning, w_range)

    baseline_holdout = macro_f1_with_weights(np.array([1.0, 1.0, 1.0]), probs_holdout, y_holdout)
    calibrated_holdout = macro_f1_with_weights(best_weights, probs_holdout, y_holdout)

    records.append({
        "seed": seed,
        "w_recyclable": best_weights[0],
        "w_organic": best_weights[2],
        "baseline_holdout_f1": baseline_holdout,
        "calibrated_holdout_f1": calibrated_holdout,
        "improvement": calibrated_holdout - baseline_holdout,
    })

results_df = pd.DataFrame(records)

## Ringkasan Hasil

In [ ]:
n_improved = (results_df["improvement"] > 0).sum()
n_same = (results_df["improvement"] == 0).sum()
n_worse = (results_df["improvement"] < 0).sum()

print(f"Dari {n_repeats} seed berbeda:")
print(f"  Kalibrasi LEBIH BAIK di holdout : {n_improved} seed ({n_improved/n_repeats*100:.0f}%)")
print(f"  Kalibrasi SAMA di holdout       : {n_same} seed ({n_same/n_repeats*100:.0f}%)")
print(f"  Kalibrasi LEBIH BURUK di holdout: {n_worse} seed ({n_worse/n_repeats*100:.0f}%)")

print(f"\nRata-rata baseline holdout F1  : {results_df['baseline_holdout_f1'].mean():.4f}")
print(f"Rata-rata calibrated holdout F1: {results_df['calibrated_holdout_f1'].mean():.4f}")
print(f"Rata-rata selisih (improvement) : {results_df['improvement'].mean():+.4f} (std: {results_df['improvement'].std():.4f})")

print(f"\nStabilitas bobot yang ditemukan tiap seed:")
print(f"  w_recyclable: mean={results_df['w_recyclable'].mean():.3f}, std={results_df['w_recyclable'].std():.3f}, min={results_df['w_recyclable'].min():.3f}, max={results_df['w_recyclable'].max():.3f}")
print(f"  w_organic   : mean={results_df['w_organic'].mean():.3f}, std={results_df['w_organic'].std():.3f}, min={results_df['w_organic'].min():.3f}, max={results_df['w_organic'].max():.3f}")

if n_improved / n_repeats >= 0.8 and results_df["improvement"].mean() > 0:
    print("\n>> KESIMPULAN: kalibrasi konsisten membantu di sebagian besar seed -- kenaikan F1 kemungkinan besar GENUINE, aman dipakai.")
elif n_improved / n_repeats <= 0.5:
    print("\n>> KESIMPULAN: kalibrasi TIDAK konsisten membantu -- hasil 0,9859 kemarin kemungkinan besar cuma kebetulan di split itu. JANGAN dijadikan submission final berdasarkan ini saja.")
else:
    print("\n>> KESIMPULAN: hasilnya di tengah-tengah -- kalibrasi mungkin membantu sedikit, tapi tidak sekuat yang terlihat di satu split kemarin. Pertimbangkan bobot yang lebih konservatif (mendekati rata-rata di atas, bukan hasil satu seed).")

## Kalibrasi Final yang Lebih Konservatif (Bobot Median dari 50 Seed)

Rentang pencarian tidak dilebarkan lagi -- `w_organic` yang terus mentok di batas bawah tiap kali
rentang diperlebar adalah tanda mulai overfitting ke `solution.csv`. Daripada pakai bobot dari 1
seed spesifik, dipakai **median** dari 50 percobaan tadi -- lebih stabil, tidak gampang ketarik
outlier dari seed yang kebetulan ekstrem.

In [ ]:
median_weights = np.array([
    results_df["w_recyclable"].median(),
    1.0,
    results_df["w_organic"].median(),
])
print(f"Bobot median dari 50 seed: {dict(zip(class_order, median_weights.round(4)))}")

calibrated_probs_full = probs_test * median_weights
final_preds = calibrated_probs_full.argmax(axis=1)

pred_map = dict(zip(test_df["id"], final_preds))
submission = pd.read_csv(CONFIG["submission_template"])
submission["predicted"] = submission["id"].map(pred_map)
assert submission["predicted"].isna().sum() == 0, "Ada id yang tidak ter-mapping, cek ulang!"
submission["predicted"] = submission["predicted"].astype(int)

out_path = "submission_champion_calibrated_robust.csv"
submission.to_csv(out_path, index=False)
print(f"Disimpan ke: {out_path}")
print(submission["predicted"].value_counts())

print(f"\n=== Evaluasi FULL solution.csv ===")
print("F1 Macro SEBELUM kalibrasi          :", f1_score(y_true_all, probs_all.argmax(axis=1), average="macro"))
print("F1 Macro SESUDAH kalibrasi (median)  :", f1_score(y_true_all, (probs_all * median_weights).argmax(axis=1), average="macro"))

## Cek Kesehatan Per-Kelas (Penting Sebelum Finalisasi)

Makro F1 itu rata-rata 3 kelas -- bisa menutupi kalau ternyata Recyclable/Electronic naik banyak
tapi Organic diam-diam dikorbankan. Cek langsung classification report & confusion matrix-nya.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_pred_before = probs_all.argmax(axis=1)
y_pred_after = (probs_all * median_weights).argmax(axis=1)

print("=== SEBELUM kalibrasi ===")
print(classification_report(y_true_all, y_pred_before, target_names=class_order, digits=4))
print("Confusion matrix:")
print(confusion_matrix(y_true_all, y_pred_before))

print("\n=== SESUDAH kalibrasi (bobot median) ===")
print(classification_report(y_true_all, y_pred_after, target_names=class_order, digits=4))
print("Confusion matrix:")
print(confusion_matrix(y_true_all, y_pred_after))